# 数据处理

In [ ]:
import pandas as pd
import os
import shutil
from sklearn.model_selection import train_test_split

base_dir = '.'

# 读取csv文件
train_df = pd.read_csv(os.path.join(base_dir, 'train.csv'))
test_df = pd.read_csv(os.path.join(base_dir, 'test.csv'))
image_dir = os.path.join(base_dir)

# 创建训练集、测试集、验证集的文件夹
# 验证集用于在训练过程中评估模型性能，从而 prevent overfitting
train_dir = os.path.join(base_dir, 'train_images')
val_dir = os.path.join(base_dir, 'val_images')
test_dir = os.path.join(base_dir, 'test_images/images')

print('image_dir:', image_dir)
print('train_dir:', train_dir)
print('val_dir:', val_dir)
print('test_dir:', test_dir)

# 创建目录,如果目录已存在则不报错
os.makedirs(train_dir, exist_ok=True)
os.makedirs(val_dir, exist_ok=True)
os.makedirs(test_dir, exist_ok=True)

# 先从训练集中划分一部分作为验证集（20%）
# 使用sklearn的train_test_split函数
train_files, val_files, train_labels, val_labels = train_test_split(
    # train_df[image] 获取数据中的图片文件名
    # train_df['label'] 获取数据中的标签
    # test_size=0.2 表示划分训练集和验证集的比例为80%:20%
    # random_state=42 设置随机种子，保证每次划分结果一致
    # stratify=train_df['label'] 按照标签的分布进行划分
    train_df['image'], train_df['label'], test_size=0.2, random_state=42, stratify=train_df['label']
            )


In [ ]:

# # 复制图片到对应的文件夹下
# def copy_images(file_list, label_list, target_dir):
#     for file_name, label in zip(file_list, label_list):
#         src_path = os.path.join(image_dir, file_name)
#         dest_path = os.path.join(target_dir, label, file_name)
#         os.makedirs(os.path.dirname(dest_path), exist_ok=True)  # 创建标签目录
#         shutil.copy(src_path, dest_path)
# # # 复制训练集和验证集图片
# copy_images(train_files, train_labels, train_dir)
# copy_images(val_files, val_labels, val_dir)
        
# # 处理测试集
# # 测试集没有标签，直接复制图片到test
# test_target_dir = os.path.join(test_dir, 'test')
# os.makedirs(test_target_dir, exist_ok=True)
# for file_name in test_df['image']:
#     # image_dir = os.path.join(image_dir, 'images')
#     dest_path = os.path.join(test_target_dir, os.path.basename(file_name))
#     shutil.copy(file_name, dest_path)
print("数据集划分完成！")

# 数据增强

In [ ]:
from torchvision import transforms

train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    # 将PIL Image或者numpy.ndarray转化为tensor，并且归一化到[0,1]
    transforms.ToTensor(),
    # 标准化数据，数据集的均值和标准差
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 验证集和测试集不进行数据增强，只进行必要的预处理
test_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])  

# 加载数据

In [ ]:
print('image_dir:', image_dir)
print('train_dir:', train_dir)
print('val_dir:', val_dir)
print('test_dir:', test_dir)

In [ ]:
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

# 使用ImageFolder加载数据
train_dataset = ImageFolder(root = train_dir, transform=train_transforms)
val_dataset = ImageFolder(root = val_dir, transform=test_transforms)
test_dataset = ImageFolder(root = test_dir, transform=test_transforms)

# 创建DataLoader
# 定义批量大小
batch_size = 32
# 训练集打乱数据，验证集和测试集不打乱 
# num_workers为数据加载的进程数
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4)

# check 数据信息

num_classes = len(train_dataset.classes)
print(f"数据集类别数: {num_classes}")
print(f"训练集大小: {len(train_dataset)}")
print(f"验证集大小: {len(val_dataset)}")
print(f"测试集大小: {len(test_dataset)}")

# 选择以及搭建模型

In [ ]:
# 使用resNet18模型
import torch
import torch.nn as nn
import torchvision.models as models

# 检查使用GPU还是CPU或者MPS
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)

# 加载预训练的ResNet18模型
model = models.resnet50(pretrained=True)# 预训练的ResNet18模型

# 冻结参数
# 冻结参数的意思是将模型的某些层的参数设置为不可训练
# 这样在训练过程中这些层的参数不会更新
# 通常在迁移学习中会冻结预训练模型的前几层，只训练最后几层
for param in model.parameters():
    param.requires_grad = False
    

# 查看模型的最后一层的输入特征数
# num_classes 是数据集的类别数
# 替换模型的最后一层全连接层，使其输出类别数与数据集一致
model.fc = nn.Linear(model.fc.in_features, num_classes)
# model.fc.in_features 获取原始全连接层的输入特征数
print(f"最后一层的输入特征数: {model.fc.in_features}")

# 将模型移动到指定设备
model = model.to(device)
print(f"模型已移动到设备: {device}")

# 打印模型结构,查看最后一层是否替换成功
print(model.fc)


# 训练模型

In [ ]:
# 定义损失函数和优化器
import torch.optim as optim

# 定义损失函数
# 使用交叉熵损失函数 ，作用于多分类问题
criterion = nn.CrossEntropyLoss()

# 定义优化器，只优化最后一层的参数
# lr是学习率 
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

# 定义学习率调度器：可以动态调整学习率
# StepLR 每隔step_size个epoch将学习率乘以gamma
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.9)

In [ ]:
num_epochs = 20  # 训练的总epoch数

# 记录训练和验证的损失和准确率，用于后续绘图
train_losses = []
val_losses = []
val_accuracies = []

for epoch in range(num_epochs):
    # 训练阶段
    # running_loss = 0.0, 训练过程中，记录每个batch的loss
    model.train()  # 设置模型为训练模式
    running_loss = 0.0
    for images, labels in train_loader:
        # 数据发送到设备
        images, labels = images.to(device), labels.to(device)
        
        # 清零梯度
        optimizer.zero_grad()
        
        # 前向传播
        outputs = model(images)
        # 计算损失
        loss = criterion(outputs, labels)
        
        # 反向传播
        loss.backward()
        # 优化器更新参数
        optimizer.step()
        
        # 统计损失
        running_loss += loss.item() * images.size(0)
    
    # 计算一个epoch的平均损失
    epoch_train_loss = running_loss / len(train_loader.dataset)
    train_losses.append(epoch_train_loss)
    
    # 验证阶段
    model.eval()  # 设置模型为评估模式
    val_running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():  # 评估时不需要计算梯度
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            val_running_loss += loss.item() * images.size(0)
            
            # 计算准确率
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    epoch_val_loss = val_running_loss / len(val_loader.dataset)
    epoch_val_accuracy = 100 * correct / total
    val_losses.append(epoch_val_loss)
    val_accuracies.append(epoch_val_accuracy)
    
    # 跟新学习率
    scheduler.step()
    
    print(f'Epoch [{epoch+1}/{num_epochs}], '
          f'Train Loss: {epoch_train_loss:.4f}, '
          f'Val Loss: {epoch_val_loss:.4f}, '
          f'Val Acc: {epoch_val_accuracy:.2f}%')

import matplotlib.pyplot as plt

# 设置中文字体（可选，如果标签想用中文）
plt.rcParams['font.sans-serif'] = ['SimHei']  # 用来正常显示中文标签
plt.rcParams['axes.unicode_minus'] = False    # 用来正常显示负号

# 创建图表
plt.figure(figsize=(12, 5))  # 设置图表大小

# 绘制训练损失和验证损失
plt.subplot(1, 2, 1)  # 1行2列的第1个子图
plt.plot(train_losses, label='训练损失 (Training Loss)', color='blue', linewidth=2)
plt.plot(val_losses, label='验证损失 (Validation Loss)', color='red', linewidth=2)
plt.title('训练和验证损失曲线')
plt.xlabel('训练轮次 (Epoch)')
plt.ylabel('损失值 (Loss)')
plt.legend()  # 显示图例
plt.grid(True, linestyle='--', alpha=0.7)  # 添加网格线

# 绘制验证准确率
plt.subplot(1, 2, 2)  # 1行2列的第2个子图
plt.plot(val_accuracies, label='验证准确率 (Validation Accuracy)', color='green', linewidth=2)
plt.title('验证准确率曲线')
plt.xlabel('训练轮次 (Epoch)')
plt.ylabel('准确率 (%)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)

# 调整布局并显示图表
plt.tight_layout()  # 自动调整子图参数，使之填充整个图像区域
plt.show()

# 也可以单独绘制损失曲线，更清晰地观察
plt.figure(figsize=(10, 6))
plt.plot(train_losses, label='训练损失 (Training Loss)', color='blue', linewidth=2, marker='o', markersize=4)
plt.plot(val_losses, label='验证损失 (Validation Loss)', color='red', linewidth=2, marker='s', markersize=4)
plt.title('训练和验证损失曲线', fontsize=14)
plt.xlabel('训练轮次 (Epoch)', fontsize=12)
plt.ylabel('损失值 (Loss)', fontsize=12)
plt.legend(fontsize=12)
plt.grid(True, linestyle='--', alpha=0.7)
plt.xticks(range(len(train_losses)), range(1, len(train_losses) + 1))  # 设置x轴刻度为1,2,3...

# 标注最后一个点的数值（可选）
last_train_loss = train_losses[-1]
last_val_loss = val_losses[-1]
plt.annotate(f'{last_train_loss:.3f}', 
             xy=(len(train_losses)-1, last_train_loss),
             xytext=(10, 10), textcoords='offset points',
             fontsize=10, color='blue')
plt.annotate(f'{last_val_loss:.3f}', 
             xy=(len(val_losses)-1, last_val_loss),
             xytext=(10, -15), textcoords='offset points',
             fontsize=10, color='red')

plt.tight_layout()
plt.show()
